In [1]:
import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling,  calculate_default_transform
from rasterio.transform import array_bounds
from scipy.signal import convolve2d
from skimage.morphology import disk, opening, closing, remove_small_objects
from skimage.measure import label, regionprops
from scipy.ndimage import binary_fill_holes

In [2]:
def process_coherence_mask(Corr_file, threshold=0.3, min_size=1000):
    """
    Process coherence raster into a binary mask using morphological filtering.

    Steps:
        - Threshold coherence
        - Morphological opening and closing
        - Remove small objects
        - Keep only the largest connected feature
        - Fill holes in the largest feature
    """
    binary = Corr_file > threshold
    se1 = disk(6)
    se2 = disk(10)
    mask = opening(binary, se1)
    mask = opening(mask, se2)
    mask = closing(mask, se1)
    mask = closing(mask, se2)
    mask = remove_small_objects(mask, min_size=min_size)
    labeled = label(mask)
    if labeled.max() > 0:
        regions = regionprops(labeled)
        largest_region = max(regions, key=lambda r: r.area)
        mask = labeled == largest_region.label
    else:
        mask[:] = False
    mask = binary_fill_holes(mask)
    return mask.astype(bool)


def reproject_to_wgs84(array, profile):
    """
    Reproject raster array to WGS84 (EPSG:4326) keeping its data extent and resolution.
    """
    dst_crs = "EPSG:4326"
    # Compute bounds from the profile
    bounds = array_bounds(profile['height'], profile['width'], profile['transform'])
    transform, width, height = calculate_default_transform(
        profile['crs'], dst_crs, profile['width'], profile['height'],
        left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3]
    )
    dst_profile = profile.copy()
    dst_profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })
    dst_array = np.empty((height, width), dtype=np.float32)
    reproject(
        source=array,
        destination=dst_array,
        src_transform=profile['transform'],
        src_crs=profile['crs'],
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest  # or bilinear for continuous data
    )
    return dst_array, dst_profile


def Landfast_Ice_App_Strain_calc(region=None):
    """
    Analyze ASF Vertex interferograms and convert them to apparent strain values.
    Outputs are saved in WGS84.
    """
    basepath = "D:/MyDrive/Stability/RawData"
    region_map = {
        "EXWR": "Extra_Wainwright",
        "UTNQ": "UTQ-NSQ",
        "KZSO": "Kotzebue_Sound",
        "OTKZ": "Outer_Kotzebue",
        "WRUT": "Wainwright-UTQ",
        "PLWR": "Point_Lay-Wainwright",
        "PBKT": "Prudhoe_Bay-Katovik",
        "PHPL": "Pt_Hope-Pt_Lay",
        "ULSP": "Uelen-SewardPenn",
        "NQKA": "NSQ-KAK",
        "RUSS": "Russia",
        "CODT": "Colville_Delta",
        "KAMK": "Kaktovik_MK_Delta",
        "NEKS": "NE_Kotzebue_Sound",
        "RURE": "Russia_redo",
        "OLIV": "Oliver_Direct",
        "AMSS": "AMSS",
    }

    if region is None:
        raise ValueError("Please provide a region code (e.g., 'EXWR')")
    if region not in region_map:
        print(f"Region not recognized: {region}")
        return

    spath = os.path.join(basepath, region_map[region])
    datadir_appstrain = os.path.join(spath, "Strain_Files_2025")
    os.makedirs(datadir_appstrain, exist_ok=True)
    datadir_corr = os.path.join(spath, "Coherence_Masks_2025")
    os.makedirs(datadir_corr, exist_ok=True)

    file_list = [f for f in os.listdir(spath) if f.endswith("_wrapped_phase.tif")]

    for fname in file_list:
        full_fname = os.path.join(spath, fname)
        out_name = os.path.join(datadir_appstrain, "app_strain_" + fname)
        out_mask_name = os.path.join(datadir_corr, "coherence_mask_" + fname)

        # Skip processing if outputs already exist
        if os.path.exists(out_name) and os.path.exists(out_mask_name):
            print(f"Skipping {fname}, outputs already exist.")
            continue
        print(f"Now reading {full_fname}")

        with rasterio.open(full_fname) as src:
            phase = src.read(1).astype(float)
            profile = src.profile

        corr_fname = fname[:60] + "_corr.tif"
        corr_path = os.path.join(spath, corr_fname)

        # Skip processing if coherence file doesn't exist
        if not os.path.exists(corr_path):
            print(f"Skipping {fname}, coherence file {corr_fname} not found.")
            continue
            
        with rasterio.open(os.path.join(spath, corr_fname)) as src:
            Corr_file = src.read(1).astype(float)

        # Phase gradients
        xgradient = np.array([-1, 0, 0, 1])
        ygradient = np.array([[-1], [0], [0], [1]])
        phase_gradient_x = convolve2d(phase, xgradient[np.newaxis, :], mode="same")
        cmplx_gradient_x = np.exp(1j * phase_gradient_x)
        phase_dx = np.arctan2(np.imag(cmplx_gradient_x), np.real(cmplx_gradient_x))
        phase_gradient_y = convolve2d(phase, ygradient, mode="same")
        cmplx_gradient_y = np.exp(1j * phase_gradient_y)
        phase_dy = np.arctan2(np.imag(cmplx_gradient_y), np.real(cmplx_gradient_y))
        phase_gradient = np.sqrt(phase_dx**2 + phase_dy**2) / 3

        # Apparent strain
        app_strain = phase_gradient * (0.0556 / ((4 * np.pi) * 80))

        # Coherence mask
        mask = process_coherence_mask(Corr_file)
        mask_app_strain = mask * app_strain

        # Reproject outputs to WGS84
        mask_app_strain_wgs84, profile_appstrain_wgs84 = reproject_to_wgs84(mask_app_strain, profile)
        mask_wgs84, profile_mask_wgs84 = reproject_to_wgs84(mask.astype(np.float32), profile)

        # Save apparent strain
        out_name = os.path.join(datadir_appstrain, "app_strain_" + fname)
        profile_appstrain_wgs84.update(dtype=rasterio.float32, count=1, compress="lzw")
        with rasterio.open(out_name, "w", **profile_appstrain_wgs84) as dst:
            dst.write(mask_app_strain_wgs84.astype(np.float32), 1)

        # Save coherence mask
        out_mask_name = os.path.join(datadir_corr, "coherence_mask_" + fname)
        profile_mask_wgs84.update(dtype=rasterio.float32, count=1, compress="lzw")
        with rasterio.open(out_mask_name, "w", **profile_mask_wgs84) as dst:
            dst.write(mask_wgs84.astype(np.float32), 1)

        print(f"Saved: {out_name}")
        print(f"Saved: {out_mask_name}")

In [3]:
Landfast_Ice_App_Strain_calc(region="PBKT")

In [4]:
Landfast_Ice_App_Strain_calc(region="EXWR")

Skipping S1AA_20170513T171605_20170525T171605_VVP012_INT80_G_ueF_02CB_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170525T171605_20170606T171606_VVP012_INT80_G_ueF_3A46_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170606T171606_20170618T171607_VVP012_INT80_G_ueF_6AE1_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170618T171607_20170630T171607_VVP012_INT80_G_ueF_2A40_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170630T171607_20170712T171608_VVP012_INT80_G_ueF_E621_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170712T171608_20170724T171558_VVP012_INT80_G_ueF_D016_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170724T171558_20170805T171558_VVP012_INT80_G_ueF_CE4C_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170805T171558_20170817T171559_VVP012_INT80_G_ueF_CF9C_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170817T171559_20170829T171559_VVP012_INT80_G_ueF_384B_wrapped_phase.tif, outputs already

In [5]:
Landfast_Ice_App_Strain_calc(region="KAMK")

Skipping S1AA_20170521T161013_20170602T161014_VVP012_INT80_G_ueF_9164_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170602T161014_20170614T161015_VVP012_INT80_G_ueF_E853_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170614T161015_20170626T161016_VVP012_INT80_G_ueF_21A0_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170626T161016_20170708T161016_VVP012_INT80_G_ueF_7986_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170930T161020_20171012T161020_VVP012_INT80_G_ueF_2084_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171012T161020_20171024T161020_VVP012_INT80_G_ueF_DD10_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171024T161020_20171105T161020_VVP012_INT80_G_ueF_598C_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171105T161020_20171117T161020_VVP012_INT80_G_ueF_9D91_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171117T161020_20171129T161020_VVP012_INT80_G_ueF_F632_wrapped_phase.tif, outputs already

In [6]:
Landfast_Ice_App_Strain_calc(region="UTNQ")

Skipping S1AA_20170515T165915_20170527T165916_VVP012_INT80_G_ueF_88F7_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170527T165916_20170608T165917_VVP012_INT80_G_ueF_C6DF_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170608T165917_20170620T165917_VVP012_INT80_G_ueF_2661_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170620T165917_20170702T165918_VVP012_INT80_G_ueF_DCD7_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170924T165930_20171006T165931_VVP012_INT80_G_ueF_6C72_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171006T165931_20171018T165931_VVP012_INT80_G_ueF_0DFD_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171018T165931_20171030T165931_VVP012_INT80_G_ueF_4027_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171030T165931_20171111T165930_VVP012_INT80_G_ueF_AE5F_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171111T165930_20171123T165930_VVP012_INT80_G_ueF_5E1E_wrapped_phase.tif, outputs already

In [7]:
Landfast_Ice_App_Strain_calc(region="KZSO")

Skipping S1AA_20170518T172518_20170530T172519_VVP012_INT80_G_ueF_9DEB_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170530T172519_20170611T172519_VVP012_INT80_G_ueF_EB49_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170611T172519_20170623T172520_VVP012_INT80_G_ueF_69DE_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170623T172520_20170705T172521_VVP012_INT80_G_ueF_6C8A_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170822T172529_20170903T172529_VVP012_INT80_G_ueF_01AD_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170903T172529_20170915T172530_VVP012_INT80_G_ueF_D340_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170915T172530_20170927T172530_VVP012_INT80_G_ueF_30B9_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170927T172530_20171009T172530_VVP012_INT80_G_ueF_4518_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171009T172530_20171021T172531_VVP012_INT80_G_ueF_53CC_wrapped_phase.tif, outputs already

In [8]:
Landfast_Ice_App_Strain_calc(region="OTKZ")

Skipping S1AA_20170511T173313_20170523T173313_VVP012_INT80_G_ueF_62B2_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170523T173313_20170604T173314_VVP012_INT80_G_ueF_4102_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170604T173314_20170616T173315_VVP012_INT80_G_ueF_7045_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170616T173315_20170628T173316_VVP012_INT80_G_ueF_238A_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170628T173316_20170710T173316_VVP012_INT80_G_ueF_A2FE_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170710T173316_20170722T173317_VVP012_INT80_G_ueF_6039_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170722T173317_20170803T173318_VVP012_INT80_G_ueF_8C72_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170803T173318_20170815T173318_VVP012_INT80_G_ueF_319E_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170815T173318_20170827T173319_VVP012_INT80_G_ueF_A904_wrapped_phase.tif, outputs already

In [9]:
Landfast_Ice_App_Strain_calc(region="WRUT")

Skipping S1BB_20170224T035435_20170308T035435_VVP012_INT80_G_ueF_798C_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170308T035435_20170320T035436_VVP012_INT80_G_ueF_87E9_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170413T035436_20170425T035437_VVP012_INT80_G_ueF_3D6D_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170425T035437_20170507T035438_VVP012_INT80_G_ueF_CD3C_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170507T035438_20170519T035438_VVP012_INT80_G_ueF_BACE_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170519T035438_20170531T035439_VVP012_INT80_G_ueF_CADF_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170531T035439_20170612T035440_VVP012_INT80_G_ueF_414A_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170612T035440_20170624T035440_VVP012_INT80_G_ueF_01A0_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170624T035440_20170706T035441_VVP012_INT80_G_ueF_9C47_wrapped_phase.tif, outputs already

In [10]:
Landfast_Ice_App_Strain_calc(region="PLWR")

Skipping S1BB_20170129T041046_20170222T041046_VVP024_INT80_G_ueF_0A70_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170222T041046_20170306T041046_VVP012_INT80_G_ueF_289E_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170306T041046_20170318T041046_VVP012_INT80_G_ueF_6ABB_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170318T041046_20170411T041047_VVP024_INT80_G_ueF_DC41_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170411T041047_20170423T041048_VVP012_INT80_G_ueF_6A0F_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170423T041048_20170517T041049_VVP024_INT80_G_ueF_BCA1_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170517T041049_20170529T041049_VVP012_INT80_G_ueF_9CC4_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170529T041049_20170610T041050_VVP012_INT80_G_ueF_EF81_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170610T041050_20170704T041051_VVP024_INT80_G_ueF_B259_wrapped_phase.tif, outputs already

In [11]:
Landfast_Ice_App_Strain_calc(region="PBKT")

In [12]:
Landfast_Ice_App_Strain_calc(region="PHPL")

Skipping S1AA_20170511T173245_20170523T173246_VVP012_INT80_G_ueF_B2CE_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170523T173246_20170604T173247_VVP012_INT80_G_ueF_F91D_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170604T173247_20170616T173247_VVP012_INT80_G_ueF_6732_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170616T173247_20170628T173248_VVP012_INT80_G_ueF_5A31_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170628T173248_20170710T173249_VVP012_INT80_G_ueF_9FCB_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170710T173249_20170722T173249_VVP012_INT80_G_ueF_96F7_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170722T173249_20170803T173250_VVP012_INT80_G_ueF_164B_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170803T173250_20170815T173251_VVP012_INT80_G_ueF_4ADF_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170815T173251_20170827T173251_VVP012_INT80_G_ueF_68AD_wrapped_phase.tif, outputs already

In [13]:
Landfast_Ice_App_Strain_calc(region="NQKA")

Skipping S1AA_20170519T162635_20170531T162636_VVP012_INT80_G_ueF_3D6A_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170531T162636_20170612T162636_VVP012_INT80_G_ueF_D0A1_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170612T162636_20170624T162637_VVP012_INT80_G_ueF_CD7A_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170624T162637_20170706T162638_VVP012_INT80_G_ueF_D24C_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170706T162638_20170718T162638_VVP012_INT80_G_ueF_05EF_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170718T162638_20170730T162639_VVP012_INT80_G_ueF_0B8F_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170730T162639_20170811T162640_VVP012_INT80_G_ueF_CC86_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170811T162640_20170823T162640_VVP012_INT80_G_ueF_3ADB_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170823T162640_20170904T162641_VVP012_INT80_G_ueF_D390_wrapped_phase.tif, outputs already

In [14]:
Landfast_Ice_App_Strain_calc(region="RUSS")

Skipping S1AA_20160708T024059_20160720T024059_VVP012_INT80_G_ueF_DB32_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20160720T024059_20160801T024100_VVP012_INT80_G_ueF_2F64_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20160801T024100_20160813T024101_VVP012_INT80_G_ueF_A61D_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20160813T024101_20160825T024103_VVP012_INT80_G_ueF_D37A_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20160825T024103_20160906T024102_VVP012_INT80_G_ueF_F0C9_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20160930T024055_20161012T024055_VVP012_INT80_G_ueF_787B_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20161012T024055_20161024T024055_VVP012_INT80_G_ueF_6589_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20161024T024055_20161105T024055_VVP012_INT80_G_ueF_FF3F_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20161105T024055_20161117T024055_VVP012_INT80_G_ueF_C1AA_wrapped_phase.tif, outputs already

In [15]:
Landfast_Ice_App_Strain_calc(region="ULSP")

Skipping S1AA_20170516T174148_20170528T174148_VVP012_INT80_G_ueF_D3A0_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170528T174148_20170609T174149_VVP012_INT80_G_ueF_6632_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170621T174150_20170703T174151_VVP012_INT80_G_ueF_4694_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170703T174151_20170715T174151_VVP012_INT80_G_ueF_C5A4_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170715T174151_20170727T174152_VVP012_INT80_G_ueF_6CC2_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20170727T174152_20170808T174153_VVP012_INT80_G_ueF_1540_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171031T174155_20171112T174155_VVP012_INT80_G_ueF_EF5B_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171112T174155_20171124T174154_VVP012_INT80_G_ueF_C586_wrapped_phase.tif, outputs already exist.
Skipping S1AA_20171124T174154_20171206T174154_VVP012_INT80_G_ueF_E59D_wrapped_phase.tif, outputs already

In [16]:
Landfast_Ice_App_Strain_calc(region="RURE")

Skipping S1BB_20170213T175713_20170225T175713_VVP012_INT80_G_ueF_87A3_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170225T175713_20170309T175713_VVP012_INT80_G_ueF_D17C_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170402T175713_20170414T175714_VVP012_INT80_G_ueF_535B_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170414T175714_20170426T175714_VVP012_INT80_G_ueF_99A1_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170426T175714_20170508T175715_VVP012_INT80_G_ueF_45A3_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170508T175715_20170520T175716_VVP012_INT80_G_ueF_E995_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170520T175716_20170601T175716_VVP012_INT80_G_ueF_C6E7_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170601T175716_20170613T175717_VVP012_INT80_G_ueF_8B8D_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170613T175717_20170625T175718_VVP012_INT80_G_ueF_C2CE_wrapped_phase.tif, outputs already

In [17]:
Landfast_Ice_App_Strain_calc(region="OLIV")

Skipping S1AA_20170408T025731_20170502T025732_VVP024_INT80_G_ueF_29C7_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170415T033809_20170509T033811_VVP024_INT80_G_ueF_98DA_wrapped_phase.tif, outputs already exist.
Skipping S1BB_20170417T032145_20170429T032146_VVP012_INT80_G_ueF_78A9_wrapped_phase.tif, outputs already exist.
